#### Importing Libraries

In [1]:
# Importing libraries for data preparation
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

#### Loading the Dataset

In [2]:
# Loading the original product classification dataset
df = pd.read_csv("../data/raw/pricerunner_aggregate.csv")

# Displaying the first 10 records
df.head(10)

,Product ID,Product Title,Merchant ID,Cluster ID,Cluster Label,Category ID,Category Label
0,1,apple iphone 8 plus 64gb silver,1,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
1,2,apple iphone 8 plus 64 gb spacegrau,2,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
2,3,apple mq8n2b/a iphone 8 plus 64gb 5.5 12mp sim...,3,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
3,4,apple iphone 8 plus 64gb space grey,4,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
4,5,apple iphone 8 plus gold 5.5 64gb 4g unlocked ...,5,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
5,6,apple iphone 8 plus gold 5.5 64gb 4g unlocked ...,6,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
6,7,apple iphone 8 plus 64 gb space grey,7,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
7,8,apple iphone 8 plus 64gb space grey,8,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
8,9,apple iphone 8 plus 64gb space grey,9,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
9,10,apple iphone 8 plus 64gb space grey,10,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones


#### Inspecting the Dataset Structure

In [3]:
# Displaying the dataset dimensions
print(f"Number of rows: {df.shape[0]:,}")
print(f"Number of columns: {df.shape[1]}")

# Displaying column names and data types
print("\nDataset Information:")
df.info()

Number of rows: 35,311
Number of columns: 7

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 35311 entries, 0 to 35310
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Product ID       35311 non-null  int64
 1   Product Title    35311 non-null  str  
 2    Merchant ID     35311 non-null  int64
 3    Cluster ID      35311 non-null  int64
 4    Cluster Label   35311 non-null  str  
 5    Category ID     35311 non-null  int64
 6    Category Label  35311 non-null  str  
dtypes: int64(4), str(3)
memory usage: 4.9 MB


**Observations on Dataset Structure:**

- The dataset contains **35,311 product listings** and **7 columns**, consistent with the original data.
- All columns contain **35,311 non-null values**, indicating no missing data.
- The dataset includes **4 integer columns** and **3 text columns**, with leading spaces present in several column names.

#### Standardizing Column Names

In [4]:
# Standardizing column names
df.columns = df.columns.str.strip().str.replace(" ", "_")

# Displaying the standardized column names
df.columns.tolist()

['Product_ID',
 'Product_Title',
 'Merchant_ID',
 'Cluster_ID',
 'Cluster_Label',
 'Category_ID',
 'Category_Label']

#### Identifying Duplicate Records

In [5]:
# Identifying exact duplicate records
exact_duplicates = df.duplicated().sum()

# Identifying duplicate records excluding the product identifier
duplicate_listings = df.duplicated(
    subset=["Product_Title", "Merchant_ID", "Cluster_ID", "Category_ID"]
).sum()

# Identifying repeated product titles
repeated_titles = df.duplicated(subset=["Product_Title"]).sum()

# Displaying duplicate counts
print(f"Exact duplicate records: {exact_duplicates:,}")
print(f"Duplicate listings excluding Product_ID: {duplicate_listings:,}")
print(f"Repeated product title occurrences: {repeated_titles:,}")

Exact duplicate records: 0
Duplicate listings excluding Product_ID: 0
Repeated product title occurrences: 4,318


**Observations on Duplicate Records:**

- No exact duplicate records or duplicate listings excluding `Product_ID` were identified.
- **4,318 repeated product title occurrences** were detected, reflecting identical titles across different product records.
- Repeated titles will be retained at this stage, with their cluster relationships considered during leakage-aware data splitting.

#### Inspecting Product Title Whitespace

In [6]:
# Identifying leading and trailing whitespace
leading_whitespace = df["Product_Title"].str.match(r"^\s").sum()
trailing_whitespace = df["Product_Title"].str.contains(r"\s$").sum()

# Identifying consecutive whitespace characters
repeated_whitespace = df["Product_Title"].str.contains(r"\s{2,}").sum()

# Identifying empty or whitespace-only titles
empty_titles = df["Product_Title"].str.strip().eq("").sum()

# Displaying whitespace-related findings
print(f"Titles with leading whitespace: {leading_whitespace:,}")
print(f"Titles with trailing whitespace: {trailing_whitespace:,}")
print(f"Titles with repeated internal whitespace: {repeated_whitespace:,}")
print(f"Empty or whitespace-only titles: {empty_titles:,}")

Titles with leading whitespace: 0
Titles with trailing whitespace: 0
Titles with repeated internal whitespace: 0
Empty or whitespace-only titles: 0


**Observations on Product Title Whitespace:**

- No product titles contain leading, trailing, or repeated internal whitespace.
- No empty or whitespace-only product titles were identified.
- No whitespace corrections are required, allowing us to preserve the original product descriptions.

#### Inspecting Product Title Capitalization

In [7]:
# Identifying titles containing uppercase letters
uppercase_titles = df["Product_Title"].str.contains(r"[A-Z]").sum()

# Identifying titles containing lowercase letters
lowercase_titles = df["Product_Title"].str.contains(r"[a-z]").sum()

# Calculating capitalization percentages
uppercase_percentage = uppercase_titles / len(df) * 100

# Displaying capitalization findings
print(f"Titles containing uppercase letters: {uppercase_titles:,}")
print(f"Titles containing lowercase letters: {lowercase_titles:,}")
print(f"Percentage containing uppercase letters: {uppercase_percentage:.2f}%")

Titles containing uppercase letters: 0
Titles containing lowercase letters: 35,311
Percentage containing uppercase letters: 0.00%


**Observations on Product Title Capitalization:**

- All **35,311 product titles** contain lowercase letters.
- No uppercase characters were identified across the dataset.
- Lowercase normalization is unnecessary, allowing the existing text to be retained without modification.

#### Identifying Conflicting Category Labels

In [8]:
# Identifying product titles associated with multiple category labels
title_category_counts = (
    df.groupby("Product_Title")["Category_Label"]
    .nunique()
)

# Selecting titles with conflicting category assignments
conflicting_titles = title_category_counts[
    title_category_counts > 1
].index

# Retrieving the affected records
conflicting_records = (
    df[df["Product_Title"].isin(conflicting_titles)]
    [["Product_ID", "Product_Title", "Cluster_ID", "Category_ID", "Category_Label"]]
    .sort_values("Product_Title")
    .reset_index(drop=True)
)

# Displaying the conflicting records
print(f"Conflicting product titles: {len(conflicting_titles):,}")
print(f"Affected records: {len(conflicting_records):,}")

display(conflicting_records)

Conflicting product titles: 2
Affected records: 4


,Product_ID,Product_Title,Cluster_ID,Category_ID,Category_Label
0,43365,blomberg kgm9550 fridge freezer,46085,2622,Fridge Freezers
1,47235,blomberg kgm9550 fridge freezer,47395,2623,Fridges
2,34378,bosch kur15a50gb integrated undercounter fridge,42819,2620,Washing Machines
3,43394,bosch kur15a50gb integrated undercounter fridge,46113,2623,Fridges


**Observations on Conflicting Category Labels:**

- **2 product titles**, affecting **4 records**, have inconsistent category assignments.
- `blomberg kgm9550 fridge freezer` is classified under both `Fridge Freezers` and `Fridges`.
- `bosch kur15a50gb integrated undercounter fridge` is classified under both `Washing Machines` and `Fridges`, indicating a likely labeling error.